# FlyCeNN vs SmolLM2-135M — full Transformer-stack replacement probe

This notebook asks a harder question than the earlier toy Transformer test:

> **Can a recurrent nonlinear network constrained by real Drosophila connectome topology replace the Transformer stack of `HuggingFaceTB/SmolLM2-135M`?**

The experiment keeps the **SmolLM2 tokenizer and pretrained token embedding / LM head initialization**, removes all Transformer blocks from the student, inserts a **FlyCeNN recurrent core**, and distills it from the frozen SmolLM2-135M teacher on FineWeb-Edu.

It evaluates:
- held-out cross-entropy and perplexity vs. original SmolLM2-135M,
- teacher→student KL,
- biological FlyWire topology vs. degree-preserving rewired control,
- parameter count,
- prefill/decode throughput,
- peak VRAM,
- sample generations.

This is a **research probe**, not evidence that a fly connectome already replaces Transformers. A short Colab run mainly tests whether the architecture learns at all and whether the biological topology gives measurable gain. Strong conclusions require matched-token / matched-compute runs with multiple seeds.

**Connectome source:** processed biological graph derived from FlyWire whole-brain connectome v783, Zenodo DOI `10.5281/zenodo.21549559`.


In [ ]:
#@title 1. Install and clone TinyCeNN-LM
import os, sys, subprocess, pathlib, importlib

REPO_DIR = pathlib.Path("/content/TinyCeNN-LM")
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "origin/main"], check=True)
else:
    subprocess.run(
        ["git", "clone", "https://github.com/vtavakkoli/TinyCeNN-LM.git", str(REPO_DIR)],
        check=True,
    )

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.56", "datasets>=3.0", "accelerate>=1.0",
    "huggingface_hub>=0.34", "pandas>=2.0", "requests>=2.31", "tqdm>=4.66"
], check=True)

print("Repository:", REPO_DIR)


In [ ]:
#@title 2. Experiment configuration
import math, time, json, random, gc, hashlib
from dataclasses import dataclass, asdict
from contextlib import nullcontext

import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = "HuggingFaceTB/SmolLM2-135M"

# "quick": architecture sanity probe; "strong": longer distillation run.
RUN_MODE = "quick"  #@param ["quick", "strong"]
SEQ_LEN = 128       #@param {type:"integer"}
BATCH_SIZE = 1      #@param {type:"integer"}
NODES = 1024        #@param {type:"integer"}
MAX_EDGES = 8192    #@param {type:"integer"}
INNER_STEPS = 1     #@param {type:"integer"}
RUN_REWIRED_CONTROL = True  #@param {type:"boolean"}

if RUN_MODE == "quick":
    TRAIN_UPDATES = 300
    GRAD_ACCUM = 2
    EVAL_BATCHES = 16
else:
    TRAIN_UPDATES = 2500
    GRAD_ACCUM = 4
    EVAL_BATCHES = 32

LR_CORE = 3e-4
LR_EMBED = 2e-5
WEIGHT_DECAY = 0.01
TEMPERATURE = 2.0
CE_WEIGHT = 0.45
KL_WEIGHT = 0.55
SEED = 123

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
dtype = (
    torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported()
    else torch.float16 if device.type == "cuda"
    else torch.float32
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("device:", device, "dtype:", dtype)
print("mode:", RUN_MODE, "updates:", TRAIN_UPDATES)


## Why this comparison is different

The original SmolLM2-135M is the **pretrained teacher and deployment baseline**. The FlyCeNN student is not allowed to keep Transformer attention or FFN blocks. It receives only:

`token embedding → input projection → recurrent FlyWire graph dynamics → output projection → tied LM head`

This makes the experiment much stricter than replacing one attention layer.

To avoid claiming that *any sparse recurrent graph* is special, the notebook can train a second student with the **same number of nodes, edges, weights, parameters, optimizer and data**, but with edge destinations shuffled. Because the destination multiset is preserved, the control keeps the same in-degree counts and source out-degree counts while destroying biological wiring motifs.


In [ ]:
#@title 3. Download and extract a real FlyWire v783 subgraph
CACHE = pathlib.Path("/content/flycenn_cache")
CACHE.mkdir(exist_ok=True)
GRAPH_FILE = CACHE / "connections_biological.csv.gz"
GRAPH_URL = (
    "https://zenodo.org/records/21549559/files/"
    "connections_biological.csv.gz?download=1"
)

if not GRAPH_FILE.exists():
    with requests.get(GRAPH_URL, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(GRAPH_FILE, "wb") as f, tqdm(
            total=total, unit="B", unit_scale=True, desc="FlyWire processed graph"
        ) as bar:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))

print("graph file:", GRAPH_FILE, f"{GRAPH_FILE.stat().st_size/2**20:.1f} MiB")

USECOLS = ["pre_root_id", "post_root_id", "syn_count"]
degree = pd.Series(dtype=np.float64)

# First pass: score high-connectivity candidate neurons.
for ch in tqdm(
    pd.read_csv(
        GRAPH_FILE, usecols=USECOLS, compression="gzip", chunksize=1_000_000
    ),
    desc="degree pass",
):
    w = np.log1p(ch["syn_count"].to_numpy(np.float64))
    a = pd.Series(w, index=ch["pre_root_id"].to_numpy()).groupby(level=0).sum()
    b = pd.Series(w, index=ch["post_root_id"].to_numpy()).groupby(level=0).sum()
    degree = degree.add(a, fill_value=0).add(b, fill_value=0)

candidate_ids = set(int(x) for x in degree.nlargest(max(NODES * 4, 4096)).index)

# Second pass: keep edges among strong candidate neurons.
parts = []
for ch in tqdm(
    pd.read_csv(
        GRAPH_FILE, usecols=USECOLS, compression="gzip", chunksize=1_000_000
    ),
    desc="edge pass",
):
    q = ch[
        ch["pre_root_id"].isin(candidate_ids)
        & ch["post_root_id"].isin(candidate_ids)
    ]
    if len(q):
        parts.append(q)

edges = (
    pd.concat(parts, ignore_index=True)
    .groupby(["pre_root_id", "post_root_id"], as_index=False)["syn_count"].sum()
    .sort_values("syn_count", ascending=False)
)

# Greedily choose NODES neurons occurring in strongest connections.
selected, seen = [], set()
for row in edges.head(MAX_EDGES * 12).itertuples(index=False):
    for x in (int(row.pre_root_id), int(row.post_root_id)):
        if x not in seen:
            seen.add(x)
            selected.append(x)
        if len(selected) >= NODES:
            break
    if len(selected) >= NODES:
        break

selected = selected[:NODES]
selected_set = set(selected)
edges = edges[
    edges["pre_root_id"].isin(selected_set)
    & edges["post_root_id"].isin(selected_set)
].head(MAX_EDGES).copy()

active_ids = sorted(
    set(edges["pre_root_id"].astype(int)) | set(edges["post_root_id"].astype(int))
)
id_to_idx = {rid: i for i, rid in enumerate(active_ids)}

src_np = edges["pre_root_id"].astype(int).map(id_to_idx).to_numpy(np.int64)
dst_np = edges["post_root_id"].astype(int).map(id_to_idx).to_numpy(np.int64)
syn_np = edges["syn_count"].to_numpy(np.float32)
N = len(active_ids)

# Normalize incoming strength so recurrent aggregation starts stable.
raw = np.log1p(syn_np).astype(np.float32)
incoming = np.zeros(N, dtype=np.float32)
np.add.at(incoming, dst_np, raw)
base_np = raw / np.maximum(incoming[dst_np], 1e-6)

rng = np.random.default_rng(SEED + 1)
dst_rewired_np = dst_np.copy()
rng.shuffle(dst_rewired_np)  # preserves destination frequency / in-degree counts

print(f"selected biological subgraph: {N:,} nodes, {len(src_np):,} directed edges")
print("synapse count range:", int(syn_np.min()), "→", int(syn_np.max()))


In [ ]:
#@title 4. Load SmolLM2-135M teacher and tokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

teacher = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=dtype if device.type == "cuda" else torch.float32,
).to(device)
teacher.eval()
teacher.config.use_cache = True
for p in teacher.parameters():
    p.requires_grad_(False)

VOCAB = teacher.config.vocab_size
HIDDEN = teacher.config.hidden_size

print("vocab:", VOCAB)
print("hidden:", HIDDEN)
print("teacher layers:", teacher.config.num_hidden_layers)
print("teacher params:", f"{sum(p.numel() for p in teacher.parameters())/1e6:.2f}M")


In [ ]:
#@title 5. Build identical FineWeb-Edu train/eval token blocks
DATASET = "HuggingFaceFW/fineweb-edu"
DATASET_CONFIG = "sample-10BT"

def token_blocks(seed, needed, seq_len):
    ds = load_dataset(
        DATASET,
        name=DATASET_CONFIG,
        split="train",
        streaming=True,
    ).shuffle(seed=seed, buffer_size=2048)
    eos = tokenizer.eos_token_id
    buf = []
    produced = 0
    for row in ds:
        text = str(row.get("text", "")).strip()
        if not text:
            continue
        ids = tokenizer(text, add_special_tokens=False)["input_ids"]
        if not ids:
            continue
        buf.extend(ids)
        buf.append(eos)
        while len(buf) >= seq_len + 1 and produced < needed:
            yield torch.tensor(buf[: seq_len + 1], dtype=torch.long)
            del buf[: seq_len + 1]
            produced += 1
        if produced >= needed:
            return

needed_train = TRAIN_UPDATES * GRAD_ACCUM * BATCH_SIZE
train_blocks = list(token_blocks(SEED + 10, needed_train, SEQ_LEN))
eval_blocks = list(token_blocks(SEED + 999, EVAL_BATCHES * BATCH_SIZE, SEQ_LEN))

def make_batches(blocks, batch_size):
    return [
        torch.stack(blocks[i:i+batch_size])
        for i in range(0, len(blocks) - batch_size + 1, batch_size)
    ]

train_batches = make_batches(train_blocks, BATCH_SIZE)
eval_batches = make_batches(eval_blocks, BATCH_SIZE)

print("train sequences:", len(train_blocks), "eval sequences:", len(eval_blocks))
print("tokens used for student training:", len(train_blocks) * SEQ_LEN)


## FlyCeNN recurrence

For each token, the student updates a persistent connectome state \(h_t\). The graph sparsity pattern is fixed, while synaptic gains, decay, self-feedback and gates are trainable.

A simplified update is:

\[
m_i = \sum_{j\to i}
\hat a_{ji}\,\tanh(g_{ji})\,h_j
\]

\[
\tilde h_t =
\tanh\left(
W_{\text{in}} e_t + m(h_{t-1}) + s\odot h_{t-1}
\right)
\]

\[
h_t = \alpha\odot h_{t-1} + (1-\alpha)\odot\tilde h_t
\]

The resulting state is projected back to SmolLM2 hidden width and decoded with the tied token embedding. There is **no self-attention, MLP Transformer block, or KV cache** in the student.


In [ ]:
#@title 6. Define the full Transformer-free FlyCeNN student
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps
    def forward(self, x):
        z = x.float()
        z = z * torch.rsqrt(z.pow(2).mean(-1, keepdim=True) + self.eps)
        return (z.to(x.dtype) * self.weight.to(x.dtype))

class FlyCeNNLM(nn.Module):
    def __init__(
        self,
        embedding_weight,
        src,
        dst,
        base_weight,
        inner_steps=1,
    ):
        super().__init__()
        vocab, hidden = embedding_weight.shape
        self.vocab = vocab
        self.hidden = hidden
        self.n_nodes = int(max(src.max(), dst.max())) + 1
        self.inner_steps = inner_steps

        self.embed = nn.Embedding(vocab, hidden)
        with torch.no_grad():
            self.embed.weight.copy_(embedding_weight.float())
        # tied LM head via F.linear(..., self.embed.weight)

        self.in_proj = nn.Linear(hidden, self.n_nodes, bias=False)
        self.out_proj = nn.Linear(self.n_nodes, hidden, bias=False)
        self.norm = RMSNorm(hidden)

        self.register_buffer("src", torch.as_tensor(src, dtype=torch.long))
        self.register_buffer("dst", torch.as_tensor(dst, dtype=torch.long))
        self.register_buffer("base_weight", torch.as_tensor(base_weight, dtype=torch.float32))

        # arctanh(0.55) gives a useful nonzero initial effective gain.
        self.edge_gain = nn.Parameter(
            torch.full((len(src),), float(np.arctanh(0.55)), dtype=torch.float32)
        )
        self.decay_logit = nn.Parameter(torch.full((self.n_nodes,), 0.3))
        self.self_gain = nn.Parameter(torch.full((self.n_nodes,), 0.15))
        self.mix_gate = nn.Parameter(torch.full((self.n_nodes,), -0.5))

    def recurrent_mix(self, state):
        # state: [B, N]
        w = self.base_weight.to(state.dtype) * torch.tanh(self.edge_gain).to(state.dtype)
        messages = state.index_select(1, self.src) * w.unsqueeze(0)
        mixed = torch.zeros_like(state)
        mixed.index_add_(1, self.dst, messages)
        return mixed

    def step(self, token_ids, state=None):
        emb = self.embed(token_ids)
        drive = torch.tanh(self.in_proj(emb))

        if state is None:
            state = torch.zeros(
                token_ids.size(0), self.n_nodes,
                device=token_ids.device, dtype=drive.dtype
            )

        decay = (0.55 + 0.40 * torch.sigmoid(self.decay_logit)).to(drive.dtype)
        gate = torch.sigmoid(self.mix_gate).to(drive.dtype)

        h = state
        for _ in range(self.inner_steps):
            mixed = self.recurrent_mix(h)
            proposal = torch.tanh(
                drive + mixed + self.self_gain.to(drive.dtype) * h
            )
            candidate = decay * h + (1.0 - decay) * proposal
            h = h + gate * (candidate - h)

        hidden = self.norm(self.out_proj(h))
        logits = F.linear(hidden, self.embed.weight)
        return logits, h, hidden

    def forward(self, input_ids, state=None, return_hidden=False):
        logits, states, hidden_seq = [], [], []
        h = state
        for t in range(input_ids.size(1)):
            lg, h, hd = self.step(input_ids[:, t], h)
            logits.append(lg)
            if return_hidden:
                hidden_seq.append(hd)
        out = torch.stack(logits, dim=1)
        if return_hidden:
            return out, h, torch.stack(hidden_seq, dim=1)
        return out, h

teacher_embed = teacher.get_input_embeddings().weight.detach().cpu()

bio_student = FlyCeNNLM(
    teacher_embed, src_np, dst_np, base_np, inner_steps=INNER_STEPS
)
rewired_student = FlyCeNNLM(
    teacher_embed, src_np, dst_rewired_np, base_np, inner_steps=INNER_STEPS
)

# Fair topology ablation: identical learned-parameter initialization.
# We intentionally do NOT copy graph buffers (dst differs by design).
with torch.no_grad():
    bio_params = dict(bio_student.named_parameters())
    for name, p in rewired_student.named_parameters():
        p.copy_(bio_params[name])

def count_params(m):
    return sum(p.numel() for p in m.parameters())

teacher_params = sum(p.numel() for p in teacher.parameters())
print("SmolLM2-135M:", f"{teacher_params/1e6:.2f}M")
print("FlyCeNN:", f"{count_params(bio_student)/1e6:.2f}M")
print("compression:", f"{teacher_params/count_params(bio_student):.2f}x fewer parameters")

# Structural check: the student must not contain Transformer attention classes.
for mod in bio_student.modules():
    name = mod.__class__.__name__.lower()
    assert "attention" not in name and "transformer" not in name
print("structural check passed: no Transformer/attention module in FlyCeNN student")


In [ ]:
#@title 7. Distillation utilities
def amp_context():
    if device.type != "cuda":
        return nullcontext()
    return torch.autocast("cuda", dtype=dtype)

def causal_ce(student_logits, ids):
    # student sees ids[:, :-1]; logits[t] predicts ids[t+1]
    return F.cross_entropy(
        student_logits.reshape(-1, student_logits.size(-1)),
        ids[:, 1:].reshape(-1),
    )

def distill_kl(student_logits, teacher_logits, temperature=TEMPERATURE):
    s = student_logits.float() / temperature
    t = teacher_logits.float() / temperature
    return (
        F.kl_div(
            F.log_softmax(s, dim=-1),
            F.softmax(t, dim=-1),
            reduction="batchmean",
        )
        * (temperature ** 2)
        / max(student_logits.size(1), 1)
    )

@torch.no_grad()
def teacher_targets(ids):
    out = teacher(input_ids=ids, use_cache=False, return_dict=True)
    # Positions 0..T-2 predict tokens 1..T-1.
    return out.logits[:, :-1].detach()

def make_optimizer(model):
    embed_ids = {id(model.embed.weight)}
    core, embed = [], []
    for p in model.parameters():
        (embed if id(p) in embed_ids else core).append(p)
    return torch.optim.AdamW(
        [
            {"params": core, "lr": LR_CORE},
            {"params": embed, "lr": LR_EMBED},
        ],
        weight_decay=WEIGHT_DECAY,
    )

def train_student(model, name):
    model = model.to(device)
    model.train()
    opt = make_optimizer(model)
    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
    losses = []
    t0 = time.perf_counter()
    consumed = 0

    pbar = tqdm(range(TRAIN_UPDATES), desc=name)
    for update in pbar:
        opt.zero_grad(set_to_none=True)
        update_loss = 0.0

        for micro in range(GRAD_ACCUM):
            batch_idx = update * GRAD_ACCUM + micro
            ids = train_batches[batch_idx].to(device, non_blocking=True)
            inp = ids[:, :-1]

            with torch.no_grad(), amp_context():
                t_logits = teacher_targets(ids)

            with amp_context():
                s_logits, _ = model(inp)
                ce = causal_ce(s_logits, ids)
                kl = distill_kl(s_logits, t_logits)
                loss = (CE_WEIGHT * ce + KL_WEIGHT * kl) / GRAD_ACCUM

            scaler.scale(loss).backward()
            update_loss += float(loss.detach()) * GRAD_ACCUM
            consumed += inp.numel()

        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()

        losses.append(update_loss)
        if update % 20 == 0 or update == TRAIN_UPDATES - 1:
            pbar.set_postfix(loss=f"{update_loss:.3f}")

    if device.type == "cuda":
        torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    return model, {
        "final_train_loss": float(np.mean(losses[-20:])),
        "train_tokens": int(consumed),
        "train_tokens_s": float(consumed / dt),
    }


In [ ]:
#@title 8. Train biological FlyCeNN
bio_student, bio_train = train_student(bio_student, "FlyCeNN biological")
print(json.dumps(bio_train, indent=2))


In [ ]:
#@title 9. Optional topology control: same model, rewired graph
rewired_train = None
if RUN_REWIRED_CONTROL:
    # Fair restart: same initialization seed where possible, topology is the intended difference.
    torch.manual_seed(SEED)
    rewired_student, rewired_train = train_student(
        rewired_student, "FlyCeNN rewired"
    )
    print(json.dumps(rewired_train, indent=2))
else:
    print("rewired control skipped")


In [ ]:
#@title 10. Held-out quality evaluation
@torch.no_grad()
def evaluate_teacher():
    losses = []
    tokens = 0
    for cpu_ids in tqdm(eval_batches, desc="SmolLM2 eval"):
        ids = cpu_ids.to(device)
        with amp_context():
            out = teacher(input_ids=ids, labels=ids, use_cache=False, return_dict=True)
        losses.append(float(out.loss.float()))
        tokens += ids.numel() - ids.size(0)
    ce = float(np.mean(losses))
    return {
        "ce": ce,
        "perplexity": float(math.exp(min(ce, 20))),
        "eval_tokens": tokens,
    }

@torch.no_grad()
def evaluate_student(model, name):
    model.eval()
    losses, kls = [], []
    tokens = 0
    for cpu_ids in tqdm(eval_batches, desc=f"{name} eval"):
        ids = cpu_ids.to(device)
        inp = ids[:, :-1]
        with amp_context():
            s_logits, _ = model(inp)
            t_logits = teacher_targets(ids)
            ce = causal_ce(s_logits, ids)
            kl = distill_kl(s_logits, t_logits)
        losses.append(float(ce.float()))
        kls.append(float(kl.float()))
        tokens += inp.numel()
    ce = float(np.mean(losses))
    return {
        "ce": ce,
        "perplexity": float(math.exp(min(ce, 20))),
        "teacher_kl": float(np.mean(kls)),
        "eval_tokens": tokens,
    }

quality = {
    "SmolLM2-135M": evaluate_teacher(),
    "FlyCeNN biological": evaluate_student(bio_student, "biological"),
}
if RUN_REWIRED_CONTROL:
    quality["FlyCeNN rewired"] = evaluate_student(rewired_student, "rewired")

display(pd.DataFrame(quality).T)


In [ ]:
#@title 11. Inference speed and VRAM: prefill + recurrent decode
def reset_cuda_peak():
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

@torch.no_grad()
def benchmark_teacher(seq_len=128, decode_steps=32, repeats=5):
    ids = torch.randint(0, VOCAB, (1, seq_len), device=device)
    reset_cuda_peak()

    # warmup
    with amp_context():
        out = teacher(input_ids=ids, use_cache=True, return_dict=True)
    if device.type == "cuda":
        torch.cuda.synchronize()
    reset_cuda_peak()

    t0 = time.perf_counter()
    past = None
    for _ in range(repeats):
        with amp_context():
            out = teacher(input_ids=ids, use_cache=True, return_dict=True)
        past = out.past_key_values
    if device.type == "cuda":
        torch.cuda.synchronize()
    prefill_dt = time.perf_counter() - t0
    prefill_tps = repeats * seq_len / prefill_dt
    prefill_mem = (
        torch.cuda.max_memory_allocated() / 2**20 if device.type == "cuda" else float("nan")
    )

    next_id = out.logits[:, -1].argmax(-1, keepdim=True)
    reset_cuda_peak()
    t0 = time.perf_counter()
    for _ in range(decode_steps):
        with amp_context():
            out = teacher(
                input_ids=next_id,
                past_key_values=past,
                use_cache=True,
                return_dict=True,
            )
        past = out.past_key_values
        next_id = out.logits[:, -1].argmax(-1, keepdim=True)
    if device.type == "cuda":
        torch.cuda.synchronize()
    decode_tps = decode_steps / (time.perf_counter() - t0)
    decode_mem = (
        torch.cuda.max_memory_allocated() / 2**20 if device.type == "cuda" else float("nan")
    )
    return {
        "prefill_tokens_s": prefill_tps,
        "prefill_peak_vram_mb": prefill_mem,
        "decode_tokens_s": decode_tps,
        "decode_peak_vram_mb": decode_mem,
    }

@torch.no_grad()
def benchmark_fly(model, seq_len=128, decode_steps=32, repeats=5):
    model.eval()
    ids = torch.randint(0, VOCAB, (1, seq_len), device=device)
    reset_cuda_peak()

    with amp_context():
        _, state = model(ids)
    if device.type == "cuda":
        torch.cuda.synchronize()
    reset_cuda_peak()

    t0 = time.perf_counter()
    state = None
    last_logits = None
    for _ in range(repeats):
        with amp_context():
            last_logits, state = model(ids, state=None)
    if device.type == "cuda":
        torch.cuda.synchronize()
    prefill_tps = repeats * seq_len / (time.perf_counter() - t0)
    prefill_mem = (
        torch.cuda.max_memory_allocated() / 2**20 if device.type == "cuda" else float("nan")
    )

    next_id = last_logits[:, -1].argmax(-1)
    reset_cuda_peak()
    t0 = time.perf_counter()
    for _ in range(decode_steps):
        with amp_context():
            lg, state, _ = model.step(next_id, state)
        next_id = lg.argmax(-1)
    if device.type == "cuda":
        torch.cuda.synchronize()
    decode_tps = decode_steps / (time.perf_counter() - t0)
    decode_mem = (
        torch.cuda.max_memory_allocated() / 2**20 if device.type == "cuda" else float("nan")
    )
    return {
        "prefill_tokens_s": prefill_tps,
        "prefill_peak_vram_mb": prefill_mem,
        "decode_tokens_s": decode_tps,
        "decode_peak_vram_mb": decode_mem,
    }

perf = {
    "SmolLM2-135M": benchmark_teacher(SEQ_LEN),
    "FlyCeNN biological": benchmark_fly(bio_student, SEQ_LEN),
}
if RUN_REWIRED_CONTROL:
    perf["FlyCeNN rewired"] = benchmark_fly(rewired_student, SEQ_LEN)

display(pd.DataFrame(perf).T)


In [ ]:
#@title 12. Context-memory scaling
lengths = [64, 128, 256, 512]
scaling_rows = []

for L in lengths:
    try:
        tr = benchmark_teacher(seq_len=L, decode_steps=8, repeats=2)
        fl = benchmark_fly(bio_student, seq_len=L, decode_steps=8, repeats=2)
        scaling_rows.append({
            "context": L,
            "smol_prefill_tps": tr["prefill_tokens_s"],
            "fly_prefill_tps": fl["prefill_tokens_s"],
            "smol_prefill_vram_mb": tr["prefill_peak_vram_mb"],
            "fly_prefill_vram_mb": fl["prefill_peak_vram_mb"],
        })
    except RuntimeError as e:
        print("context", L, "skipped:", e)
        if device.type == "cuda":
            torch.cuda.empty_cache()

scaling = pd.DataFrame(scaling_rows)
display(scaling)


In [ ]:
#@title 13. Side-by-side generations
PROMPTS = [
    "Artificial intelligence can",
    "The city of Vienna is",
    "A neural network learns",
    "The future of efficient computing",
]

@torch.no_grad()
def generate_fly(model, prompt, max_new_tokens=48, temperature=0.8):
    model.eval()
    ids = tokenizer(prompt, return_tensors="pt")["input_ids"].to(device)
    with amp_context():
        logits, state = model(ids)
    out = ids[0].tolist()
    next_logits = logits[:, -1]

    for _ in range(max_new_tokens):
        probs = F.softmax(next_logits.float() / temperature, dim=-1)
        nxt = torch.multinomial(probs, 1).squeeze(1)
        out.append(int(nxt.item()))
        with amp_context():
            next_logits, state, _ = model.step(nxt, state)
        if int(nxt.item()) == tokenizer.eos_token_id:
            break
    return tokenizer.decode(out, skip_special_tokens=True)

for prompt in PROMPTS:
    ids = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        teacher_out = teacher.generate(
            **ids,
            max_new_tokens=48,
            do_sample=True,
            temperature=0.8,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )
    print("=" * 100)
    print("PROMPT:", prompt)
    print("\nSMOLLM2-135M:\n", tokenizer.decode(teacher_out[0], skip_special_tokens=True))
    print("\nFLYCENN:\n", generate_fly(bio_student, prompt))


In [ ]:
#@title 14. Final scientific summary and save results
rows = []
for name in quality:
    row = {
        "model": name,
        "params_m": (
            teacher_params / 1e6
            if name == "SmolLM2-135M"
            else count_params(bio_student) / 1e6
        ),
        **quality[name],
        **perf[name],
    }
    rows.append(row)

summary = pd.DataFrame(rows)

teacher_ce = quality["SmolLM2-135M"]["ce"]
bio_ce = quality["FlyCeNN biological"]["ce"]

report = {
    "base_model": BASE_MODEL,
    "run_mode": RUN_MODE,
    "config": {
        "seq_len": SEQ_LEN,
        "batch_size": BATCH_SIZE,
        "nodes": N,
        "edges": int(len(src_np)),
        "inner_steps": INNER_STEPS,
        "train_updates": TRAIN_UPDATES,
        "grad_accum": GRAD_ACCUM,
        "seed": SEED,
    },
    "quality": quality,
    "performance": perf,
    "train": {
        "FlyCeNN biological": bio_train,
        "FlyCeNN rewired": rewired_train,
    },
    "derived": {
        "fly_ce_gap_vs_smollm2": bio_ce - teacher_ce,
        "fly_ppl_ratio_vs_smollm2": (
            quality["FlyCeNN biological"]["perplexity"]
            / quality["SmolLM2-135M"]["perplexity"]
        ),
        "parameter_compression_x": teacher_params / count_params(bio_student),
        "decode_speed_ratio_fly_over_smollm2": (
            perf["FlyCeNN biological"]["decode_tokens_s"]
            / perf["SmolLM2-135M"]["decode_tokens_s"]
        ),
        "decode_vram_ratio_fly_over_smollm2": (
            perf["FlyCeNN biological"]["decode_peak_vram_mb"]
            / perf["SmolLM2-135M"]["decode_peak_vram_mb"]
            if perf["SmolLM2-135M"]["decode_peak_vram_mb"] > 0 else None
        ),
    },
}

if RUN_REWIRED_CONTROL:
    report["derived"]["biological_topology_ce_gain"] = (
        quality["FlyCeNN rewired"]["ce"] - quality["FlyCeNN biological"]["ce"]
    )
    report["derived"]["biological_topology_ppl_gain_pct"] = 100 * (
        quality["FlyCeNN rewired"]["perplexity"]
        - quality["FlyCeNN biological"]["perplexity"]
    ) / quality["FlyCeNN rewired"]["perplexity"]

OUT = REPO_DIR / "results" / "flycenn_smollm2_135m"
OUT.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUT / "summary.csv", index=False)
scaling.to_csv(OUT / "context_scaling.csv", index=False)
(OUT / "report.json").write_text(json.dumps(report, indent=2))

# Save only the trained Transformer-free student state + graph metadata.
torch.save(
    {
        "model_state": {k: v.detach().cpu() for k, v in bio_student.state_dict().items()},
        "active_root_ids": active_ids,
        "config": report["config"],
        "base_model": BASE_MODEL,
    },
    OUT / "flycenn_student.pt",
)

display(summary)
print(json.dumps(report["derived"], indent=2))
print("\nSaved to:", OUT)

print("\nINTERPRETATION")
print("1) First check CE/perplexity: a short quick run is expected to trail the pretrained teacher.")
print("2) If biological_topology_ce_gain > 0, the biological wiring beat the rewired control.")
print("3) Check decode speed + VRAM: FlyCeNN has fixed recurrent state and no KV cache.")
print("4) If learning is stable, switch RUN_MODE='strong' before changing the architecture.")
